**1. SOLIS:** Palaiž report ģenerēšanu uz dev kopas konkrētam jautājumam (baseline)

**2. SOLIS:** Analizē nepareizās atbildes

**3. SOLIS:** Labo uzvedni un saglabā to atmiņā (mainīgajā)

**4. SOLIS:** Atkārtoti ģenerē un salīdzina akuritāti ar baseline

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
from IPython.display import display
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.azure_openai import AzureOpenAI
from tqdm import tqdm
from dotenv import load_dotenv
import yaml
from pathlib import Path
import re as re
load_dotenv()

from scripts.extractmd import Extractor
from scripts.vectorindex import QnAEngine
from scripts.utilities import (
    get_prompt_dict, get_questions, get_answers,
    get_procurement_content, get_config_data, get_ini_files,
    get_supplementary_info
)
from scripts.gen_results import gen_results

In [ ]:
# Jautājuma nr., kuru vēlas uzlabot:
target_q = '39.1'

# Jautājumi, kas jāapstrādā:
questions_to_process = ['39', '39.1']

config_dir_name = 'dev_config'

# Esošs CSV kā baseline — ja norādīts, 1. SOLIS tiek izlaists.
existing_baseline_csv = None #'reports/_test_jaut_nr.39.33_dev_02.05/report.csv'

In [ ]:
embedding_conf = {
    'embeddingmodel': 'BAAI/bge-m3',
    'chunk_size': 1536,
    'chunk_overlap': 0,
    'top_similar': 5,
    'n4rerank': 0,
    'use_similar_chunks': True,
    'prevnext': True
}
embedding = HuggingFaceEmbedding(model_name=embedding_conf['embeddingmodel'],trust_remote_code=True)

llm = AzureOpenAI(
    azure_deployment='gpt-4o',
    azure_endpoint=os.environ.get('AZURE_ENDPOINT', ''),
    temperature=0.0,
    api_version=os.environ.get('AZURE_OPENAI_VERSION', ''),
    api_key=os.environ.get('AZURE_OPENAI_KEY', ''),
    timeout=120, max_retries=3, top_p=0.0001
)

extractor = Extractor()

**FILE SETTINGS**

In [ ]:
script_dir = globals()['_dh'][0]

question_file_path   = script_dir / 'questions' / 'questions.yaml'
prompt_file          = script_dir / 'questions' / 'prompts.yaml'
report_dir           = script_dir / 'reports'
config_dir           = script_dir / config_dir_name
procurement_file_dir = script_dir / 'cfla_files'
answer_file_dir      = script_dir / 'answers'

# Reports mapei
new_q        = target_q.replace('.', '_')
baseline_dir  = report_dir / f'opt_{new_q}_baseline'
baseline_csv  = baseline_dir / 'report.csv'
candidate_dir = report_dir / f'opt_{new_q}_candidate'
candidate_csv = candidate_dir / 'report.csv'

if existing_baseline_csv is not None:
    _p = Path(existing_baseline_csv)
    baseline_csv = _p if _p.is_absolute() else script_dir / _p
    if baseline_csv.exists():
        print(f'Esošs baseline CSV: {baseline_csv}')
        print('1. soļa ģenerēšana tiks izlaista.')
    else:
        print(f'Norādītais fails neeksistē: {baseline_csv}')
        print('Pārbaudiet ceļu vai iestatiet existing_baseline_csv = None.')
else:
    print(f'Baseline CSV tiks ģenerēts: {baseline_csv}')

# Ielādē datus
question_dictionary = get_questions(question_file_path)
prompt_dictionary   = get_prompt_dict(prompt_file, question_dictionary)
supplementary_info  = get_supplementary_info()
ini_files = get_ini_files(config_dir, True, baseline_csv)
print(f'Apstrādājamie faili ({len(ini_files)}): {sorted(ini_files)}')

**PALĪGFUNKCIJAS**

In [ ]:
def find_question_data(q_dict, nr):
    for q in q_dict:
        if str(q['nr']) == str(nr):
            return q
        for sq in q.get('questions', []):
            if str(sq['nr']) == str(nr):
                return sq
    return None

def compute_accuracy(csv_path, q_nr):
    if not csv_path.exists():
        return None, 0
    df = pd.read_csv(csv_path)
    df['Atbilde'] = df['Atbilde'].astype(str).str.strip().str.lower()
    df['Sagaidama_atbilde'] = df['Sagaidāmā atbilde'].astype(str).str.strip().str.lower()
    df = df[(df['Nr'].astype(str) == str(q_nr)) & (df['Sagaidama_atbilde'] != '?')]
    if len(df) == 0:
        return None, 0
    correct = (df['Atbilde'] == df['Sagaidama_atbilde']).sum()
    return round(correct / len(df) * 100, 2), len(df)

def count_usage_in_file(q_list, pid, key):
    """Skaita, cik jautājumi lieto doto prompt_id."""
    count = 0
    for q in q_list:
        if q.get(key) == pid:
            count += 1
        count += count_usage_in_file(q.get('questions', []), pid, key)
    return count

def update_q_prompt_id(q_list, nr, key, new_pid):
    """Atrod jautājumu un nomaina tā prompt_key."""
    base = str(nr)[:-2] if str(nr).endswith('-0') else str(nr)
    for q in q_list:
        if str(q.get('nr', '')) == base:
            q[key] = new_pid
            return True
        if update_q_prompt_id(q.get('questions', []), nr, key, new_pid):
            return True
    return False

async def run_report(out_dir, out_csv, prompt_dict, label=''):
    out_dir.mkdir(parents=True, exist_ok=True)
    if out_csv.exists():
        out_csv.unlink()

    print(f'Sāk ģenerēšanu: {label}')
    all_rows = []

    for file in tqdm(sorted(ini_files), desc='Config files', unit='file'):
        configfile = config_dir / f'{file}.ini'
        tqdm.write(f'Apstrada: {configfile}')

        _, proc_file, agr_file, ans_file = get_config_data(
            configfile, procurement_file_dir, answer_file_dir
        )
        answer_dict = get_answers(ans_file)
        content = get_procurement_content(extractor, proc_file, agr_file)

        engine = QnAEngine(embedding, llm)
        await engine.createIndex(
            content, 'Procurement',
            chunk_size=embedding_conf['chunk_size'],
            chunk_overlap=embedding_conf['chunk_overlap']
        )

        rows = gen_results(
            engine, configfile, embedding_conf,
            question_dictionary, answer_dict,
            prompt_dict, supplementary_info,
            questions_to_process
        )
        for r in rows:
            r.insert(0, file)
        all_rows.extend(rows)

    cols = ['Iepirkuma ID', 'Nr', 'Atbilde', 'Sagaidāmā atbilde', 'Pamatojums', 'Uzvedne']
    df = pd.DataFrame(all_rows, columns=cols)
    df.to_csv(out_csv, index=False, encoding='utf-8')

    acc, n = compute_accuracy(out_csv, target_q)
    print(f'\n[{label}] Jautājums {target_q}: akuritāte = {acc}% ({n} iepirkumi)')
    return df

**1. SOLIS: Baseline ģenerēšana**

In [ ]:
# Palaiž report ģenerēšanu ar pašreizējo uzvedni uz visiem ini_files
if existing_baseline_csv is not None:
    print(f'Baseline ģenerēšana izlaista. Izmanto esošu CSV: {baseline_csv}')
    baseline_df = None
else:
    baseline_df = await run_report(baseline_dir, baseline_csv, prompt_dictionary, 'Baseline')

**2. SOLIS: Kļūdu analīze**

In [ ]:
# Parāda visas atbildes target_q. Zaļas = pareizas, sarkanas = nepareizas.
def show_failures(csv_path, q_nr):
    df = pd.read_csv(csv_path)
    df['Atbilde'] = df['Atbilde'].astype(str).str.strip().str.lower()
    df['Sagaidāmā atbilde'] = df['Sagaidāmā atbilde'].astype(str).str.strip().str.lower()
    df = df[(df['Nr'].astype(str) == str(q_nr)) & (df['Sagaidāmā atbilde'] != '?')]
    df = df.copy()
    df['Pareizi'] = df['Atbilde'] == df['Sagaidāmā atbilde']

    acc, n = compute_accuracy(csv_path, q_nr)
    print(f'Jautajums {q_nr} — kopa: {n}, '
          f'pareizi: {df["Pareizi"].sum()}, '
          f'nepareizi: {(~df["Pareizi"]).sum()}, '
          f'akuritāte: {acc}%')

    def highlight(row):
        color = '#d4edda' if row['Pareizi'] else '#f8d7da'
        return [f'background-color: {color}'] * len(row)

    cols_show = ['Iepirkuma ID', 'Sagaidāmā atbilde', 'Atbilde', 'Pamatojums', 'Pareizi']
    display(df[cols_show].style.apply(highlight, axis=1))

show_failures(baseline_csv, target_q)

**Konteksta apskate**

In [ ]:
# inspect_file = 'SNP202131'  # Jānorāda fails ar nepareizu atbildi

# configfile = config_dir / f'{inspect_file}.ini'
# _, proc_file, agr_file, _ = get_config_data(
#     configfile, procurement_file_dir, answer_file_dir
# )
# content_inspect = get_procurement_content(extractor, proc_file, agr_file)

# engine_inspect = QnAEngine(embedding, llm)
# await engine_inspect.createIndex(
#     content_inspect, 'Procurement',
#     chunk_size=embedding_conf['chunk_size'],
#     chunk_overlap=embedding_conf['chunk_overlap']
# )

# q_data = find_question_data(question_dictionary, target_q)
# q_text = q_data.get('question', q_data.get('question0', '')) if q_data else ''
# print(f'Jautajums: {q_text}')
# print('=' * 50)

# nodes = engine_inspect.getSimilarNodes(
#     q_text, embedding_conf['top_similar'], embedding_conf['prevnext']
# )
# for i, (txt, score, meta) in enumerate(
#     zip(nodes['text'], nodes['score'], nodes['metadata'])
# ):
#     print(f'\n Fragments {i+1} (score: {score:.4f})')
#     print(f'Metadata: {meta}')
#     print(txt)

**3. SOLIS: uzvednes labošana**

In [ ]:
df_err = pd.read_csv(baseline_csv)
df_err['Atbilde'] = df_err['Atbilde'].astype(str).str.strip().str.lower()
df_err['Sagaidāmā atbilde'] = df_err['Sagaidāmā atbilde'].astype(str).str.strip().str.lower()
df_err = df_err[
    (df_err['Nr'].astype(str) == str(target_q)) &
    (df_err['Sagaidāmā atbilde'] != '?')
]
failures = df_err[df_err['Atbilde'] != df_err['Sagaidāmā atbilde']]

if failures.empty:
    print('Kļūdu nav. Uzvednei nav nepieciešami uzlabojumi.')
    new_prompt = prompt_dictionary.get(target_q, prompt_dictionary.get('0', ''))
else:
    q_data     = find_question_data(question_dictionary, target_q)
    q_text     = q_data.get('question', q_data.get('question0', '')) if q_data else ''
    cur_prompt = prompt_dictionary.get(target_q, prompt_dictionary.get('0', ''))

    fail_cols    = ['Iepirkuma ID', 'Sagaidāmā atbilde', 'Atbilde', 'Pamatojums']
    failures_str = failures[fail_cols].to_string(index=False)

    suggestion_request = (
        'Tu esi eksperts uzvedņu inženierijā. Palīdzi uzlabot LLM uzvedni'
        'iepirkuma dokumentācijas pārbaudes sistēmai.\n\n'
        'Pašreizējā uzvedne:\n' + cur_prompt + '\n\n'
        'Jautājums, kas LLM tiek dots kopā ar uzvedni:\n' + q_text + '\n\n'
        'Nepareizās atbildes (Iepirkuma ID | Sagaidāmā atbilde | LLM atbilde | LLM skaidrojums):\n'
        + failures_str + '\n\n'
        'Uzdevums:\n'
        '1. Analizē, kāpēc LLM kļūdās katrā no gadījumiem.\n'
        '2. Iesaki konkrētas izmaiņas uzvednes tekstā.\n'
        '3. Ja vajadzīgs, pievieno skaidrus nosacījumus "ja", "nē" un "n/a" gadījumiem.\n'
        '4. Atbildi latviski.\n'
        '5. Gatavoto uzvednes tekstu ievieto tieši starp tagiem <suggested_prompt> un </suggested_prompt>'
    )

    response = llm.complete(suggestion_request)
    print('LLM ierosinājums:\n')
    print(response.text)

    match = re.search(r'<suggested_prompt>(.*?)</suggested_prompt>', response.text, re.DOTALL)
    if match:
        new_prompt = match.group(1).strip()
        print('\nIeteiktā uzvedne ievietota mainīgajā new_prompt.')
    else:
        new_prompt = cur_prompt
        print('\nLLM neatgrieza uzvedni <suggested_prompt> tagos.')
        print('new_prompt paliek nemainīts. Rediģējiet to manuāli nākamajā šūnā.')

In [ ]:
# new_prompt = """Šeit var ievietot manuālu uzvedni, ja LLM ieteikums neder."""

print('Uzvedne, kas tiks izmantota candidate report ģenerēšanai:\n\n')
print(new_prompt)

In [ ]:
# Testē jauno uzvedni
candidate_prompt_dict = dict(prompt_dictionary)
candidate_prompt_dict[target_q] = new_prompt
print(f'Uzvedne jautājumam {target_q} nomainīta atmiņā (mainīgajā).')

**4. SOLIS: Atkārtota ģenerēšana ar jauno uzvedni**

In [ ]:
candidate_df = await run_report(candidate_dir, candidate_csv, candidate_prompt_dict, 'Candidate')

**Salīdzinājums: Baseline vs Candidate**

In [ ]:
def compare_results(base_csv, cand_csv, q_nr):
    acc_b, n_b = compute_accuracy(base_csv, q_nr)
    acc_c, n_c = compute_accuracy(cand_csv, q_nr)

    if acc_b is not None and acc_c is not None:
        delta = round(acc_c - acc_b, 2)
        sign  = '+' if delta >= 0 else ''
    else:
        delta, sign = None, ''

    print('=' * 50)
    print(f'Jautājuma nr: {q_nr}')
    print('=' * 50)
    print(f'  Baseline:   {acc_b}%  ({n_b} iepirkumi)')
    print(f'  Candidate:  {acc_c}%  ({n_c} iepirkumi)')
    print(f'  Izmaiņas:    {sign}{delta}%')
    print('=' * 50)

    df_b = pd.read_csv(base_csv)
    df_c = pd.read_csv(cand_csv)

    for df in [df_b, df_c]:
        df['Atbilde'] = df['Atbilde'].astype(str).str.strip().str.lower()
        df['Sagaidāmā atbilde'] = df['Sagaidāmā atbilde'].astype(str).str.strip().str.lower()

    df_b = df_b[
        (df_b['Nr'].astype(str) == str(q_nr)) &
        (df_b['Sagaidāmā atbilde'] != '?')
    ].copy()
    df_c = df_c[df_c['Nr'].astype(str) == str(q_nr)][['Iepirkuma ID', 'Atbilde', 'Pamatojums']]
    df_c = df_c.rename(columns={'Atbilde': 'Candidate', 'Pamatojums': 'Candidate pamatojums'})

    merged = df_b[['Iepirkuma ID', 'Sagaidāmā atbilde', 'Atbilde', 'Pamatojums']].copy()
    merged = merged.rename(columns={'Atbilde': 'Baseline', 'Pamatojums': 'Baseline pamatojums'})
    merged = merged.merge(df_c, on='Iepirkuma ID', how='left')

    merged['B'] = merged['Baseline'] == merged['Sagaidāmā atbilde']
    merged['C'] = merged['Candidate'] == merged['Sagaidāmā atbilde']

    def hl(row):
        styles = [''] * len(row)
        for col_name, mark_col in [('Baseline', 'B'), ('Candidate', 'C')]:
            if col_name in row.index:
                idx = row.index.get_loc(col_name)
                styles[idx] = (
                    'background-color: #d4edda'
                    if row[mark_col]
                    else 'background-color: #f8d7da'
                )
        return styles

    cols_display = [
        'Iepirkuma ID', 'Sagaidāmā atbilde',
        'Baseline', 'Candidate', 'B', 'C',
        'Baseline pamatojums', 'Candidate pamatojums'
    ]
    display(merged[cols_display].style.apply(hl, axis=1))


compare_results(baseline_csv, candidate_csv, target_q)

**Saglabā uzvedni**

Ja Candidate akurātums pārsniedz Baseline akurātumu, uzvedne tiek automātiski atjaunināta `prompts.yaml` failā.

In [ ]:
acc_b, _ = compute_accuracy(baseline_csv, target_q)
acc_c, _ = compute_accuracy(candidate_csv, target_q)

print('=' * 60)
print(f'Jautājums:  {target_q}')
print(f'Baseline:   {acc_b}%')
print(f'Candidate:  {acc_c}%')
print('=' * 60)

if acc_b is None or acc_c is None:
    print('KĻŪDA: Nav rezultātu. Palaidiet baseline un candidate ģenerēšanu!')

elif acc_c <= acc_b:
    print(f'Candidate ({acc_c}%) nav labāks par baseline ({acc_b}%).')
    print('Uzvedne netiek saglabāta. Nav izmaiņu failos.')

else:
    delta = round(acc_c - acc_b, 2)
    print(f'Candidate ir labāks par +{delta}% — saglabā uzvedni...')

    is_q0 = str(target_q).endswith('-0')
    prompt_key = 'prompt0-id' if is_q0 else 'prompt-id'

    q_data_target = find_question_data(question_dictionary, target_q)
    if q_data_target is None:
        print(f'KĻŪDA: Jautājums {target_q} nav atrasts.')
    else:
        current_prompt_id = q_data_target.get(prompt_key)
        if not current_prompt_id:
            print(f'KĻŪDA: Jautājumam {target_q} nav "{prompt_key}" lauks.')
        else:
            with open(prompt_file, 'r', encoding='utf-8') as f:
                prompts_data = yaml.safe_load(f) or []

            is_default = any(p.get('id') == current_prompt_id and p.get('default') for p in prompts_data)

            # BaseLoader: visus skaitļus patur kā string, lai, piemēram, 39.20 nepārvēršas par 39.2
            with open(question_file_path, 'r', encoding='utf-8') as f:
                questions_data_fresh = yaml.load(f, Loader=yaml.BaseLoader) or []

            usage = count_usage_in_file(questions_data_fresh, current_prompt_id, prompt_key)
            print(f'  Uzvedne "{current_prompt_id}" — lietojums: {usage} jautājumos, '
                  f'noklusējums: {is_default}.')

            if not is_default and usage <= 1:
                found = False
                for p in prompts_data:
                    if p['id'] == current_prompt_id:
                        p['prompt'] = new_prompt
                        found = True
                        break
                if not found:
                    prompts_data.append({'id': current_prompt_id, 'prompt': new_prompt})

                with open(prompt_file, 'w', encoding='utf-8') as f:
                    yaml.dump(prompts_data, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

                print(f'  [OK] prompts.yaml "{current_prompt_id}" atjaunināta.')

            else:
                # Koplietota uzvedne — izveido jaunu ID
                reason = 'noklusējums' if is_default else f'koplietota ({usage} jautājumi)'
                print(f'  Veido jaunu ID ({reason})...')

                safe_nr = str(target_q).replace('.', '_').replace('-', '_')
                new_id = f'p_for_{safe_nr}'
                existing_ids = {p['id'] for p in prompts_data}
                if new_id in existing_ids:
                    v = 2
                    while f'{new_id}_v{v}' in existing_ids:
                        v += 1
                    new_id = f'{new_id}_v{v}'

                prompts_data.append({'id': new_id, 'prompt': new_prompt})
                with open(prompt_file, 'w', encoding='utf-8') as f:
                    yaml.dump(prompts_data, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

                print(f'  [OK] prompts.yaml jauna uzvedne "{new_id}" pievienota.')

                # Atjaunina questions.yaml tikai šim jautājumam
                if update_q_prompt_id(questions_data_fresh, target_q, prompt_key, new_id):
                    with open(question_file_path, 'w', encoding='utf-8') as f:
                        yaml.dump(
                            questions_data_fresh, f,
                            allow_unicode=True, default_flow_style=False, sort_keys=False
                        )
                    print(f'  [OK] questions.yaml — jautājumam {target_q} "{prompt_key}" = "{new_id}".')
                else:
                    print(f'  Jautājums {target_q} nav atrasts questions.yaml!')

            print(f'\n  Jauns akurātums: {acc_c}% (bija: {acc_b}%, pieaugums: +{delta}%)')
